# 05 与冻结三状态基准逐日比较

本 Notebook 将 03 号 Notebook 生成的五列 CSV 中的 `date / three_state`，与远端冻结基准文件 `IC_1545_three_state_and_downside_warning.csv` 的 `date / three_state` 对齐比较。

只比较原始三状态，不使用基准文件的 `downside_warning` 列。日期按实际开盘生效日比较，不再次平移。

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 60)

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "pool_runner.py").is_file()
)
GENERATED_PATH = Path(
    os.environ.get(
        "1545_SIGNAL_CSV_PATH",
        str(PACKAGE_ROOT / "IC_1545_frozen_exit_signals.csv"),
    )
).expanduser()
BASELINE_PATH = Path(
    os.environ.get(
        "1545_BASELINE_THREE_STATE_PATH",
        "/home/hzy/cta/三状态冻结/IC_1545_three_state_and_downside_warning.csv",
    )
).expanduser()

if not GENERATED_PATH.is_file():
    raise FileNotFoundError(f"找不到生成文件: {GENERATED_PATH}")

generated = pd.read_csv(GENERATED_PATH)
if not {"date", "three_state"}.issubset(generated.columns):
    raise ValueError(f"生成文件必须包含 date/three_state: {list(generated.columns)}")
generated = generated[["date", "three_state"]].copy()
generated["date"] = pd.to_datetime(generated["date"], errors="raise").dt.normalize()
generated["three_state"] = pd.to_numeric(generated["three_state"], errors="raise").astype(int)
generated = generated.drop_duplicates("date", keep="last").sort_values("date")

print("生成文件:", GENERATED_PATH.resolve())
print("生成文件日期:", generated["date"].min().date(), "->", generated["date"].max().date())
print("基准文件:", BASELINE_PATH)
if not BASELINE_PATH.is_file():
    print("\n本机未找到基准文件。远端运行时请确认该路径存在；本格不把‘缺失基准’误报为一致。")
    display(pd.DataFrame([{
        "status": "BASELINE_NOT_FOUND",
        "baseline_path": str(BASELINE_PATH),
        "generated_rows": len(generated),
    }]))
else:
    baseline = pd.read_csv(BASELINE_PATH)
    needed = {"date", "three_state"}
    if not needed.issubset(baseline.columns):
        raise ValueError(f"基准文件必须包含 date/three_state: {list(baseline.columns)}")
    baseline = baseline[["date", "three_state"]].copy()
    baseline["date"] = pd.to_datetime(baseline["date"], errors="raise").dt.normalize()
    baseline["three_state"] = pd.to_numeric(baseline["three_state"], errors="raise").astype(int)
    baseline = baseline.drop_duplicates("date", keep="last").sort_values("date")

    common = generated.merge(
        baseline,
        on="date",
        how="inner",
        suffixes=("_generated", "_baseline"),
    )
    common["match"] = common["three_state_generated"].eq(common["three_state_baseline"])
    common["delta_generated_minus_baseline"] = (
        common["three_state_generated"] - common["three_state_baseline"]
    )
    mismatches = common.loc[~common["match"]].copy()

    summary = pd.DataFrame([{
        "generated_rows": len(generated),
        "baseline_rows": len(baseline),
        "common_dates": len(common),
        "matching_dates": int(common["match"].sum()),
        "mismatch_dates": int((~common["match"]).sum()),
        "match_rate": float(common["match"].mean()) if len(common) else np.nan,
        "generated_date_min": generated["date"].min().date().isoformat(),
        "generated_date_max": generated["date"].max().date().isoformat(),
        "baseline_date_min": baseline["date"].min().date().isoformat(),
        "baseline_date_max": baseline["date"].max().date().isoformat(),
    }])
    print("比较摘要：")
    display(summary.round(6))

    print("共同日期上的状态列联表（行=生成，列=基准）：")
    display(pd.crosstab(
        common["three_state_generated"],
        common["three_state_baseline"],
        rownames=["generated_three_state"],
        colnames=["baseline_three_state"],
        dropna=False,
    ))

    print("逐状态计数：")
    state_compare = pd.DataFrame({
        "generated_days": generated["three_state"].value_counts().reindex([-1, 0, 1], fill_value=0),
        "baseline_days": baseline["three_state"].value_counts().reindex([-1, 0, 1], fill_value=0),
    }).rename_axis("three_state").reset_index()
    state_compare["delta_generated_minus_baseline"] = (
        state_compare["generated_days"] - state_compare["baseline_days"]
    )
    display(state_compare)

    if mismatches.empty:
        print("COMPARE_STATUS: 完全一致（共同 date 上 three_state 没有差异）")
    else:
        print(f"COMPARE_STATUS: 存在 {len(mismatches)} 个共同日期差异")
        print("全部差异日期：")
        display(mismatches[[
            "date",
            "three_state_generated",
            "three_state_baseline",
            "delta_generated_minus_baseline",
        ]])

    print("COMPARE_THREE_STATE_END")